# Tests for `encryp.ipynb`

Every code cell tagged **`test`** is one piece of a single pytest module. The
runner at the bottom stitches them together and hands them to pytest, so
fixtures, `parametrize` and assertion rewriting all behave normally.

## How to run

- **In Jupyter** — Run All. The last cell prints the pytest report.
- **Only some tests** — `run_tests("-k", "raw_data", "-v")`.
- **Headless** — `jupyter nbconvert --execute --to notebook --stdout
  ver_1/test_encryp.ipynb`.

## What is covered

| Area | Guards against |
| --- | --- |
| Stage 0 validation | A container of the wrong type being walked past, leaving raw values in output that looks clean |
| Stage 1 classification | A host's kind being guessed when the record states it; the wrong user path being missed |
| Stage 1 codecs | A value that cannot be reversed; a file path that loses its filename |
| Stage 2 detection | Raw CII surviving in `raw_data`; public and private addresses being confused |
| Stage 2 codecs | Two values sharing one placeholder |
| The gateway | Half-processed output escaping; an invented token being decrypted |
| Stage 3 round trip | Ciphertext containing `_` being re-split wrongly; padding collisions |
| Issuance and audit | Partial commits; refused attempts going unaudited |
| Key rotation | Old tokens becoming undecryptable; a token stamped with the wrong key |

## Notes

- The tests reach into `namespace`, which is the executed `library` cells of
  `encryp.ipynb`. Edit a cell there, re-run here, and the change is picked up.
- `Data/ocsf_edr_mock.ndjson` carries no `raw_data` field, so the tests that
  cover it use the connector's real record shape as an inline fixture rather
  than the corpus.

## Loading the library cells

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import pytest


def locate_notebook(name: str = "encryp.ipynb") -> Path:
    """Find a notebook from a Jupyter kernel, a script, or a temp module.

    Inside Jupyter there is no __file__, and the runner cell below assembles
    these cells into a module outside this directory, so the path is resolved
    at run time instead of being assumed. ENCRYP_NOTEBOOK is honoured only
    for the notebook it actually names, so pointing it at the engine does not
    hide the test notebook from the runner.
    """
    override = os.environ.get("ENCRYP_NOTEBOOK")
    if override and Path(override).name == name:
        return Path(override)
    roots = [Path(__file__).parent] if "__file__" in globals() else []
    roots += [Path.cwd(), Path.cwd() / "ver_1"]
    for root in roots:
        candidate = root / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"could not locate {name} from {Path.cwd()}")


NOTEBOOK = locate_notebook()


def load_library_cells() -> str:
    """Return the source of the notebook cells tagged 'library', in order.

    Selecting by tag rather than by index keeps the tests working when cells
    are reordered, and keeps scratch cells out of the test run.
    """
    with NOTEBOOK.open(encoding="utf-8") as notebook_file:
        notebook = json.load(notebook_file)
    code_cells = [cell for cell in notebook["cells"] if cell["cell_type"] == "code"]
    tagged = [cell for cell in code_cells if "library" in cell["metadata"].get("tags", [])]
    if not tagged:
        raise AssertionError("encryp.ipynb has no code cell tagged 'library'")
    return "\n".join("".join(cell["source"]) for cell in tagged)


namespace = {}
exec(compile(load_library_cells(), str(NOTEBOOK), "exec"), namespace)

ALPHABET = namespace["ALPHABET"]
PAD_CHAR = namespace["PAD_CHAR"]
MAX_SUFFIX_LEN = namespace["MAX_SUFFIX_LEN"]
PLACEHOLDER_RE = namespace["PLACEHOLDER_RE"]
PREFIX_CODEC = namespace["PREFIX_CODEC"]
CiiPolicy = namespace["CiiPolicy"]
CiiValueTooLongError = namespace["CiiValueTooLongError"]
KeyRing = namespace["KeyRing"]
TokenCollisionError = namespace["TokenCollisionError"]
TokenHallucinationError = namespace["TokenHallucinationError"]
TokenLengthError = namespace["TokenLengthError"]
UnknownKeyVersionError = namespace["UnknownKeyVersionError"]
UnsanitizedOutputError = namespace["UnsanitizedOutputError"]
VersionedCipher = namespace["VersionedCipher"]
classify_ip = namespace["classify_ip"]
decode_cii = namespace["decode_cii"]
detokenize_log = namespace["detokenize_log"]
encode_cii = namespace["encode_cii"]
ensure_schema = namespace["ensure_schema"]
key_versions_in_use = namespace["key_versions_in_use"]
load_key_ring = namespace["load_key_ring"]
lookup_issued_token = namespace["lookup_issued_token"]
mark_cii = namespace["mark_cii"]
mark_free_text = namespace["mark_free_text"]
record_issued_token = namespace["record_issued_token"]
restore_from_llm = namespace["restore_from_llm"]
safe_decrypt = namespace["safe_decrypt"]
sanitize_for_llm = namespace["sanitize_for_llm"]
sanitize_stream = namespace["sanitize_stream"]
tokenize_field = namespace["tokenize_field"]
tokenize_log = namespace["tokenize_log"]
unmark_cii = namespace["unmark_cii"]
walk_and_tokenize = namespace["walk_and_tokenize"]

# Stage 1 and the file pipeline
CLASSIFIED_PATHS = namespace["CLASSIFIED_PATHS"]
CODECS = namespace["CODECS"]
EDR_POLICY = namespace["EDR_POLICY"]
FIELD_RULES = namespace["FIELD_RULES"]
FILE_PATH_PATHS = namespace["FILE_PATH_PATHS"]
FILE_PATH_PREFIXES = namespace["FILE_PATH_PREFIXES"]
IGNORED_VALUES = namespace["IGNORED_VALUES"]
IPV4_RE = namespace["IPV4_RE"]
NonAsciiValueError = namespace["NonAsciiValueError"]
NotOcsfError = namespace["NotOcsfError"]
OCSF_INPUT = namespace["OCSF_INPUT"]
RawCiiLeakError = namespace["RawCiiLeakError"]
classify_and_mark = namespace["classify_and_mark"]
classify_hostname = namespace["classify_hostname"]
classify_username = namespace["classify_username"]
collect_raw_cii = namespace["collect_raw_cii"]
decode_ascii = namespace["decode_ascii"]
encode_ascii = namespace["encode_ascii"]
find_leaks = namespace["find_leaks"]
mark_file_path = namespace["mark_file_path"]
parse_ocsf_line = namespace["parse_ocsf_line"]
reset_heuristic_warning = namespace["reset_heuristic_warning"]
run_pipeline = namespace["run_pipeline"]
sanitize_document = namespace["sanitize_document"]
split_file_path = namespace["split_file_path"]
OcsfValidationError = namespace["OcsfValidationError"]
choose_prefix = namespace["choose_prefix"]
validate_ocsf = namespace["validate_ocsf"]
import warnings

KEY_V1 = "F3D2FE0E66B2AA56806944AA2770D53A"
TWEAK_V1 = "6A81B1DC1D04B6"  # 56-bit tweak selects FF3-1
KEY_V2 = "2DE79D232DF5585D68CE47882AE256D6"
TWEAK_V2 = "9A768A92F60E12"

FORTIGATE_LOG = {
    "class_name": "Network Activity",
    "device": {"name": "Device-FGT-01", "uid": "DEVID-FG200ETK"},
    "src_endpoint": {"ip": "192.168.10.87", "port": "55692",
                     "interface_name": "Network-VL.2"},
    "dst_endpoint": {"ip": "94.5.151.23", "port": 80, "mac": "N/A",
                     "location": {"country": "Japan"}},
    "firewall_rule": {"name": "Policy-54", "uid": "54"},
    "unmapped": {"transip": "203.0.113.1", "vd": "root",
                 "poluuid": "0f20863f-74eb-4641-885b-aa63b8512687"},
    "traffic": {"bytes": 8320},
    "severity": "Informational",
    "raw_data": (
        'devname="Device-FGT-01" devid="DEVID-FG200ETK" srcip=192.168.10.87 '
        'srcport=55692 dstip=94.5.151.23 dstport=80 policyname="Policy-54" '
        'poluuid="0f20863f-74eb-4641-885b-aa63b8512687" transip=203.0.113.1 '
        "sentbyte=1581"
    ),
}

SANGFOR_LOG = {
    "class_name": "Network Activity",
    "src_endpoint": {"ip": "10.0.0.121", "interface_name": "Trust"},
    "dst_endpoint": {"ip": "154.226.192.239", "location": {"country": "Germany"}},
    "url": {"url_string": "http://update.microsoft.com"},
    "unmapped": {"suser": "j.doe", "PolicyUUID": "490DB181A28E4CF9898AF508FF37CF0D",
                 "SourceSystem": "public"},
    "message": "Network Activity Event",
    "raw_data": (
        "fwlog[6088269]: CEF:0|Sangfor|NGAF|AF8.0.95|6|website browsing|1|"
        "suser=j.doe src=10.0.0.121 dst=154.226.192.239 act=Allow "
        "Request=http://update.microsoft.com"
    ),
}

# Every raw value that must not survive sanitization.
FORTIGATE_CII = ["Device-FGT-01", "DEVID-FG200ETK", "192.168.10.87", "94.5.151.23",
                 "203.0.113.1", "Network-VL.2", "Policy-54",
                 "0f20863f-74eb-4641-885b-aa63b8512687"]
SANGFOR_CII = ["10.0.0.121", "154.226.192.239", "j.doe", "update.microsoft.com",
               "490DB181A28E4CF9898AF508FF37CF0D", "Trust"]

## A fake database

In [ ]:



class FakeCursor:
    """In-memory stand-in for a psycopg cursor, faithful about rowcount."""

    def __init__(self, connection):
        self.connection = connection
        self.row = None
        self.rows = []
        self.rowcount = -1

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False

    def execute(self, query, params=()):
        normalized_query = " ".join(query.split()).upper()
        self.connection.executed.append((normalized_query, params))
        self.row = None
        self.rows = []
        self.rowcount = -1

        if normalized_query.startswith("CREATE SCHEMA"):
            self.connection.schema_ready = True
        elif normalized_query.startswith("SELECT PREFIX"):
            self.row = self.connection.issued.get(params[0])
        elif normalized_query.startswith("SELECT KEY_VERSION"):
            counts = Counter(row[2] for row in self.connection.issued.values())
            self.rows = sorted(counts.items())
        elif normalized_query.startswith("INSERT INTO VAULT_SCHEMA.ISSUED_TOKENS"):
            assert "ON CONFLICT (TOKEN) DO NOTHING" in normalized_query
            token, prefix, suffix_len, key_version = params
            if token in self.connection.issued:
                self.rowcount = 0
            else:
                self.connection.issued[token] = (prefix, suffix_len, key_version)
                self.rowcount = 1
        elif normalized_query.startswith("INSERT INTO METADATA_SCHEMA.DETOKENIZE_LOG"):
            self.connection.audit_log.append(params)
            self.rowcount = 1
        else:
            raise AssertionError(f"unexpected query: {normalized_query}")

    def fetchone(self):
        return self.row

    def fetchall(self):
        return self.rows


class FakeConnection:
    def __init__(self):
        self.issued = {}
        self.audit_log = []
        self.executed = []
        self.commit_count = 0
        self.schema_ready = False

    def cursor(self):
        return FakeCursor(self)

    def commit(self):
        self.commit_count += 1

## Fixtures

In [ ]:



@pytest.fixture
def env(monkeypatch):
    """A hermetic FF3 environment holding only the v1 key."""
    for name in list(os.environ):
        if name.startswith("FF3_"):
            monkeypatch.delenv(name, raising=False)
    monkeypatch.setenv("FF3_KEY", KEY_V1)
    monkeypatch.setenv("FF3_TWEAK", TWEAK_V1)
    return monkeypatch


@pytest.fixture
def key_ring(env):
    return load_key_ring()


def rotate_to_v2(env):
    """Perform a rotation: add v2, keep v1 loaded, make v2 active."""
    env.delenv("FF3_KEY", raising=False)
    env.delenv("FF3_TWEAK", raising=False)
    env.setenv("FF3_KEY_V1", KEY_V1)
    env.setenv("FF3_TWEAK_V1", TWEAK_V1)
    env.setenv("FF3_KEY_V2", KEY_V2)
    env.setenv("FF3_TWEAK_V2", TWEAK_V2)
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v2")
    return load_key_ring()


def retire_v1(env):
    """Take v1 off the ring entirely, leaving only v2 loaded."""
    for name in ("FF3_KEY", "FF3_TWEAK", "FF3_KEY_V1", "FF3_TWEAK_V1"):
        env.delenv(name, raising=False)
    env.setenv("FF3_KEY_V2", KEY_V2)
    env.setenv("FF3_TWEAK_V2", TWEAK_V2)
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v2")
    return load_key_ring()


@pytest.fixture
def rotated(env):
    return rotate_to_v2(env)


@pytest.fixture
def connection():
    return FakeConnection()


def round_trip(connection, key_ring, original):
    token = tokenize_log(original, key_ring, connection)
    return token, safe_decrypt(connection, key_ring, token, "analyst")

## Stage 2: codecs

In [ ]:



@pytest.mark.parametrize(
    "prefix, value",
    [
        ("INT_IP", "192.168.10.87"), ("INT_IP", "10.0.0.121"), ("INT_IP", "10.0.0.1"),
        ("EXT_IP", "94.5.151.23"), ("EXT_IP", "8.8.8.8"), ("EXT_IP", "203.0.113.255"),
        ("MAC", "00:1a:2b:3c:4d:5e"),
        ("POLICY_UUID", "0f20863f-74eb-4641-885b-aa63b8512687"),
        ("POLICY_ID", "490DB181A28E4CF9898AF508FF37CF0D"),
        ("HOST", "Device-FGT-01"), ("DEVICE", "DEVID-FG200ETK"),
        ("USER", "j.doe"), ("USER", "admin"), ("IFACE", "Network-VL.2002"),
        ("POLICY", "Policy-54"), ("URL_HOST", "update.microsoft.com"),
        ("EMAIL", "a@b.co"),
    ],
)
def test_every_codec_round_trips_exactly(prefix, value):
    """Stage 2 must be reversible with nothing stored anywhere."""
    placeholder = encode_cii(prefix, value)

    assert PLACEHOLDER_RE.fullmatch(placeholder), f"not a valid placeholder: {placeholder}"
    assert decode_cii(placeholder) == value


def test_placeholder_suffix_stays_inside_the_cipher_alphabet():
    """Stage 3 can only encrypt what its alphabet covers."""
    for prefix, value in [("INT_IP", "10.0.0.1"), ("HOST", "Device-FGT-01"),
                          ("USER", "j.doe"), ("MAC", "00:1a:2b:3c:4d:5e")]:
        suffix = PLACEHOLDER_RE.fullmatch(encode_cii(prefix, value)).group(2)
        assert set(suffix) <= set(ALPHABET) - {PAD_CHAR}
        assert len(suffix) <= MAX_SUFFIX_LEN


def test_every_prefix_has_a_codec():
    """A prefix without a codec would produce an irreversible placeholder."""
    namespace["CODECS"]  # present
    for prefix, codec in PREFIX_CODEC.items():
        assert codec in namespace["CODECS"], f"{prefix} names a missing codec {codec!r}"


def test_two_uuid_formats_keep_their_own_shape():
    """Fortigate writes dashed lowercase, Sangfor bare uppercase.

    One codec cannot restore both, which is why they get separate prefixes.
    """
    dashed = "0f20863f-74eb-4641-885b-aa63b8512687"
    bare = "490DB181A28E4CF9898AF508FF37CF0D"

    assert decode_cii(encode_cii("POLICY_UUID", dashed)) == dashed
    assert decode_cii(encode_cii("POLICY_ID", bare)) == bare


def test_value_too_long_for_the_cipher_is_refused():
    with pytest.raises(CiiValueTooLongError, match="over the 36-character limit"):
        encode_cii("URL_HOST", "https://exfil-data-server.ru/upload/very/long/path")


@pytest.mark.parametrize(
    "value, expected",
    [("192.168.10.87", "INT_IP"), ("10.0.0.121", "INT_IP"), ("172.16.0.1", "INT_IP"),
     ("94.5.151.23", "EXT_IP"), ("8.8.8.8", "EXT_IP"),
     ("not-an-ip", None), ("::1", None)],
)
def test_internal_and_external_addresses_are_told_apart(value, expected):
    assert classify_ip(value) == expected

## Stage 2: detection

In [ ]:



def test_structured_fields_are_marked_by_path():
    marked = mark_cii(FORTIGATE_LOG)

    assert marked["src_endpoint"]["ip"].startswith("[INT_IP_")
    assert marked["dst_endpoint"]["ip"].startswith("[EXT_IP_")
    assert marked["device"]["name"].startswith("[HOST_")
    assert marked["device"]["uid"].startswith("[DEVICE_")
    assert marked["unmapped"]["poluuid"].startswith("[POLICY_UUID_")


def test_fields_an_analyst_needs_are_left_alone():
    marked = mark_cii(FORTIGATE_LOG)

    assert marked["src_endpoint"]["port"] == "55692"
    assert marked["dst_endpoint"]["port"] == 80
    assert marked["dst_endpoint"]["location"]["country"] == "Japan"
    assert marked["severity"] == "Informational"
    assert marked["traffic"]["bytes"] == 8320
    assert marked["firewall_rule"]["uid"] == "54"


def test_placeholder_values_that_carry_nothing_are_skipped():
    marked = mark_cii({"dst_endpoint": {"mac": "N/A"}, "unmapped": {"suser": "(null)"}})

    assert marked["dst_endpoint"]["mac"] == "N/A"
    assert marked["unmapped"]["suser"] == "(null)"


@pytest.mark.parametrize(
    "log, secrets",
    [(FORTIGATE_LOG, FORTIGATE_CII), (SANGFOR_LOG, SANGFOR_CII)],
)
def test_raw_data_never_keeps_a_value_the_structured_fields_hid(log, secrets):
    """The gap that path rules alone leave open.

    raw_data restates every value in free text, including ones no regex would
    recognise (`devname="Device-FGT-01"`).
    """
    raw_data = mark_cii(log)["raw_data"]

    leaked = [secret for secret in secrets if secret in raw_data]
    assert not leaked, f"raw_data still contains {leaked}"


def test_one_value_gets_one_identity_across_the_whole_document():
    """A model can only correlate if the same host is the same token."""
    marked = mark_cii(FORTIGATE_LOG)

    from_field = marked["src_endpoint"]["ip"]
    assert from_field in marked["raw_data"]
    assert marked["device"]["name"] in marked["raw_data"]
    assert marked["firewall_rule"]["name"] in marked["raw_data"]


def test_url_keeps_its_shape_but_loses_its_host():
    marked = mark_cii(SANGFOR_LOG)
    url = marked["raw_data"]

    assert "update.microsoft.com" not in url
    assert "Request=http://[URL_HOST_" in url


def test_free_text_sweep_finds_addresses_that_no_field_rule_covers():
    """Values living only in raw_data still have to be caught."""
    swept = mark_free_text("relay 10.0.254.1 saw aa:bb:cc:dd:ee:ff hit 1.1.1.1")

    assert "10.0.254.1" not in swept
    assert "aa:bb:cc:dd:ee:ff" not in swept
    assert "1.1.1.1" not in swept


def test_stage_2_round_trips_a_whole_document():
    for log in (FORTIGATE_LOG, SANGFOR_LOG):
        assert unmark_cii(mark_cii(log)) == log


def test_public_ips_can_be_left_readable_for_threat_context():
    policy = CiiPolicy(redact_public_ips=False)

    marked = mark_cii(FORTIGATE_LOG, policy)

    assert marked["dst_endpoint"]["ip"] == "94.5.151.23"
    assert marked["src_endpoint"]["ip"].startswith("[INT_IP_")

## The gateway

In [ ]:



@pytest.mark.parametrize(
    "log, secrets",
    [(FORTIGATE_LOG, FORTIGATE_CII), (SANGFOR_LOG, SANGFOR_CII)],
)
def test_no_cii_survives_sanitization(connection, key_ring, log, secrets):
    """The one test this whole system exists to pass."""
    llm_ready = json.dumps(sanitize_for_llm(log, key_ring, connection))

    leaked = [secret for secret in secrets if secret in llm_ready]
    assert not leaked, f"sanitized output still contains {leaked}"


@pytest.mark.parametrize("log", [FORTIGATE_LOG, SANGFOR_LOG])
def test_gateway_round_trips_the_original_document(connection, key_ring, log):
    llm_ready = sanitize_for_llm(log, key_ring, connection)

    assert restore_from_llm(llm_ready, connection, key_ring, "analyst") == log


def test_stage_2_output_alone_is_not_safe_to_send():
    """The module docstring's warning is load-bearing, so pin it down."""
    marked = mark_cii(FORTIGATE_LOG)

    assert decode_cii(marked["src_endpoint"]["ip"]) == "192.168.10.87"
    assert "192168010087" in json.dumps(marked), "the address, merely re-encoded"


def test_the_gateway_refuses_to_emit_an_untokenized_placeholder(
    connection, key_ring, monkeypatch
):
    """Neutralise Stage 3 so the guard is the only thing left standing."""
    monkeypatch.setitem(namespace, "tokenize_log", lambda node, ring, conn: node)

    with pytest.raises(UnsanitizedOutputError) as error:
        sanitize_for_llm(FORTIGATE_LOG, key_ring, connection)

    assert error.value.leaked
    assert all(PLACEHOLDER_RE.fullmatch(item) for item in error.value.leaked)


def test_marking_twice_changes_nothing(connection, key_ring):
    """mark_cii must be idempotent or a re-run would double-encode."""
    once = mark_cii(FORTIGATE_LOG)

    assert mark_cii(once) == once


def test_a_token_the_model_invented_is_refused(connection, key_ring):
    llm_ready = sanitize_for_llm(FORTIGATE_LOG, key_ring, connection)
    llm_ready["src_endpoint"]["ip"] = "[INT_IP_ZZZZZZZZZZZZ]"

    with pytest.raises(TokenHallucinationError):
        restore_from_llm(llm_ready, connection, key_ring, "analyst")

    assert connection.audit_log[-1] == ("[INT_IP_ZZZZZZZZZZZZ]", "analyst", "rejected")


def test_every_detokenization_is_audited(connection, key_ring):
    llm_ready = sanitize_for_llm(FORTIGATE_LOG, key_ring, connection)
    restore_from_llm(llm_ready, connection, key_ring, "analyst")

    assert connection.audit_log
    assert all(actor == "analyst" for _, actor, _ in connection.audit_log)
    assert {outcome for *_, outcome in connection.audit_log} == {"success"}


def test_the_same_host_is_the_same_token_in_separate_documents(connection, key_ring):
    """Global tokens: correlation across logs is the point of dropping case_id."""
    first = sanitize_for_llm({"src_endpoint": {"ip": "10.0.0.5"}}, key_ring, connection)
    second = sanitize_for_llm(
        {"dst_endpoint": {"ip": "10.0.0.5"}, "severity": "High"}, key_ring, connection
    )

    assert first["src_endpoint"]["ip"] == second["dst_endpoint"]["ip"]
    assert len(connection.issued) == 1


def test_stream_sanitizes_ndjson_record_by_record(connection, key_ring):
    lines = [json.dumps(FORTIGATE_LOG), "", json.dumps(SANGFOR_LOG)]

    out = list(sanitize_stream(lines, key_ring, connection))

    assert len(out) == 2
    for text, secrets in zip(out, (FORTIGATE_CII, SANGFOR_CII)):
        assert not [secret for secret in secrets if secret in text]


def test_gateway_commits_once_per_record(connection, key_ring):
    sanitize_for_llm(FORTIGATE_LOG, key_ring, connection)

    assert connection.commit_count == 1

## Stage 3: round trip

In [ ]:



def test_round_trip_and_prefix_is_preserved(connection, key_ring):
    token, restored = round_trip(connection, key_ring, "[HOST_01]")

    assert token.startswith("[HOST_")
    assert restored == "[HOST_01]"
    assert connection.audit_log == [(token, "analyst", "success")]


def test_round_trip_holds_across_the_whole_suffix_space(connection, key_ring):
    """Property check: nothing that goes in may fail to come back out."""
    failures = []
    for prefix in ("HOST", "INT_IP", "USER", "LOG_COLLECTOR"):
        for index in range(150):
            original = f"[{prefix}_{index:02d}]"
            try:
                _, restored = round_trip(connection, key_ring, original)
            except Exception as error:  # noqa: BLE001 - reported below
                failures.append((original, f"{type(error).__name__}: {error}"))
                continue
            if restored != original:
                failures.append((original, f"restored as {restored}"))

    assert not failures, f"{len(failures)} of 600 values did not round-trip: {failures[:5]}"


def test_ciphertext_containing_the_pad_character_still_decrypts(connection, key_ring):
    """Regression: re-parsing a token with a regex split the prefix wrongly."""
    seen_pad = 0
    for index in range(300):
        token, restored = round_trip(connection, key_ring, f"[HOST_{index:02d}]")
        if PAD_CHAR in token[len("[HOST_") :]:
            seen_pad += 1
        assert restored == f"[HOST_{index:02d}]"

    assert seen_pad, "no ciphertext contained the pad character; test proves nothing"


def test_distinct_values_never_share_a_token(key_ring):
    seen = {}
    for index in range(300):
        original = f"[HOST_{index}]"
        token = tokenize_field(key_ring, original)
        assert token not in seen, f"{original} and {seen[token]} both produced {token}"
        seen[token] = original


@pytest.mark.parametrize(
    "short, underscored",
    [("[HOST_01]", "[HOST__01]"), ("[USER_A]", "[USER__A]"), ("[HOST_1]", "[HOST___1]")],
)
def test_padding_is_never_confused_with_a_literal_underscore(key_ring, short, underscored):
    assert tokenize_field(key_ring, short) != tokenize_field(key_ring, underscored)


def test_suffix_longer_than_the_cipher_maximum_is_rejected(key_ring):
    too_long = "[HOST_" + "A" * (key_ring.active.maxLen + 1) + "]"

    with pytest.raises(TokenLengthError, match="exceeds the cipher maximum"):
        tokenize_field(key_ring, too_long)


def test_bracketed_text_that_is_not_a_placeholder_passes_through(key_ring):
    for value in ["logs[6124]", "[HOST__01]", "[1003]", "[]"]:
        assert tokenize_field(key_ring, value) == value

## Stage 3: issuance and audit

In [ ]:



def test_tokenize_log_commits_once_per_log_not_once_per_token(connection, key_ring):
    tokenize_log(
        {"a": "[HOST_01]", "b": "[USER_02]", "c": ["[INT_IP_03]", "[AGENT_04]"]},
        key_ring,
        connection,
    )

    assert len(connection.issued) == 4
    assert connection.commit_count == 1


def test_repeated_value_is_recorded_once(connection, key_ring):
    tokenize_log({"a": "[HOST_01]", "b": "[HOST_01]"}, key_ring, connection)

    assert len(connection.issued) == 1


def test_one_token_mapping_to_two_plaintexts_is_refused(connection):
    record_issued_token(connection, "[HOST_AB_CD]", "HOST", 2, "v1")

    with pytest.raises(TokenCollisionError):
        record_issued_token(connection, "[HOST_AB_CD]", "HOST_AB", 2, "v1")


def test_unissued_token_is_rejected_before_decrypt(connection, key_ring):
    with pytest.raises(TokenHallucinationError) as error:
        safe_decrypt(connection, key_ring, "[HOST_01]", "analyst")

    assert error.value.token == "[HOST_01]"


def test_refused_detokenization_is_still_audited(connection, key_ring):
    with pytest.raises(TokenHallucinationError):
        safe_decrypt(connection, key_ring, "[HOST_99]", "mallory")

    assert connection.audit_log == [("[HOST_99]", "mallory", "rejected")]
    assert connection.commit_count == 1, "the audit row must be committed, not lost"


def test_tokens_are_global_and_not_scoped_to_a_caller(connection, key_ring):
    """Dropping case_id means issuance is the only check that remains."""
    token = tokenize_log("[HOST_01]", key_ring, connection)

    assert safe_decrypt(connection, key_ring, token, "anyone-else") == "[HOST_01]"


def test_invented_placeholder_inside_a_document_is_refused(connection, key_ring):
    tokenized = tokenize_log({"host": "[HOST_01]"}, key_ring, connection)
    tokenized["host"] = "[HOST_99]"

    with pytest.raises(TokenHallucinationError):
        detokenize_log(tokenized, connection, key_ring, "analyst")

    assert connection.audit_log == [("[HOST_99]", "analyst", "rejected")]

## Key rotation

In [ ]:



def test_token_issued_before_rotation_still_decrypts(connection, env):
    token = tokenize_log("[HOST_01]", load_key_ring(), connection)

    ring = rotate_to_v2(env)
    assert ring.active_version == "v2"
    assert ring.versions == ("v1", "v2")
    assert safe_decrypt(connection, ring, token, "analyst") == "[HOST_01]"


def test_rotation_changes_the_token_issued_for_the_same_value(connection, env):
    before = tokenize_field(load_key_ring(), "[HOST_01]")
    after = tokenize_field(rotate_to_v2(env), "[HOST_01]")

    assert before != after


def test_one_log_can_mix_key_versions(connection, env):
    old = tokenize_log("[HOST_01]", load_key_ring(), connection)
    ring = rotate_to_v2(env)
    new = tokenize_log("[USER_02]", ring, connection)

    restored = detokenize_log({"a": old, "b": new}, connection, ring, "analyst")

    assert restored == {"a": "[HOST_01]", "b": "[USER_02]"}


def test_new_tokens_are_stamped_by_the_key_not_the_environment(connection, rotated, env):
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v9")  # changed after the ring was built

    tokenize_log("[HOST_01]", rotated, connection)

    assert {version for _, _, version in connection.issued.values()} == {"v2"}


def test_key_taken_off_the_ring_is_refused_and_audited(connection, env):
    token = tokenize_log("[HOST_01]", load_key_ring(), connection)

    ring = retire_v1(env)
    assert ring.versions == ("v2",)

    with pytest.raises(UnknownKeyVersionError) as error:
        safe_decrypt(connection, ring, token, "analyst")

    assert error.value.version == "v1"
    assert error.value.token == token
    assert connection.audit_log[-1] == (token, "analyst", "rejected")


def test_key_versions_in_use_reports_what_blocks_retirement(connection, env):
    tokenize_log({"a": "[HOST_01]", "b": "[USER_02]"}, load_key_ring(), connection)
    tokenize_log("[HOST_03]", rotate_to_v2(env), connection)

    assert key_versions_in_use(connection) == {"v1": 2, "v2": 1}


def test_key_ring_resolves_versions_case_insensitively(rotated):
    assert rotated.for_version("V1") is rotated.for_version("v1")


def test_active_version_must_be_on_the_ring(key_ring):
    with pytest.raises(RuntimeError, match="not on the key ring"):
        KeyRing([key_ring.active], "v9")


def test_a_provider_name_survives_issuance_and_detokenization(connection, key_ring):
    """The provider seam: the name a token is stamped with is the name the
    ring resolves."""
    ring = KeyRing([VersionedCipher("KMS-2026-01", key_ring.active.cipher)], "KMS-2026-01")

    token = tokenize_log("[HOST_01]", ring, connection)

    assert connection.issued[token][2] == "kms-2026-01"
    assert safe_decrypt(connection, ring, token, "analyst") == "[HOST_01]"


def test_key_material_is_never_rendered(key_ring):
    assert KEY_V1 not in repr(key_ring)
    assert KEY_V1 not in repr(key_ring.active)
    assert TWEAK_V1 not in repr(key_ring)

## Configuration

In [ ]:



def test_environment_variables_are_required(env):
    env.delenv("FF3_KEY")
    env.delenv("FF3_TWEAK")

    with pytest.raises(RuntimeError, match="FF3_KEY"):
        load_key_ring()

    env.setenv("FF3_KEY", "key")
    with pytest.raises(RuntimeError, match="FF3_TWEAK"):
        load_key_ring()


def test_key_version_env_name_is_not_mistaken_for_a_key(env):
    env.setenv("FF3_KEY_VERSION", "v7")

    assert load_key_ring().versions == ("v7",)


def test_a_56_bit_tweak_selects_ff3_1(env):
    """NIST kept FF3-1 after the attacks on the original FF3."""
    assert len(bytes.fromhex(TWEAK_V1)) == 7
    assert load_key_ring().active.minLen >= 2


def test_database_url_is_required(monkeypatch):
    monkeypatch.delenv("DATABASE_URL", raising=False)

    with pytest.raises(RuntimeError, match="DATABASE_URL"):
        namespace["connect_database"]()


@pytest.mark.parametrize(
    "value", ["[HOST_01]", "[INT_IP_01]", "[LOG_COLLECTOR_01]", "[HOST_A_B]"]
)
def test_pad_character_never_appears_in_a_captured_suffix(value):
    assert PAD_CHAR in ALPHABET
    match = PLACEHOLDER_RE.fullmatch(value)
    assert match is not None
    assert PAD_CHAR not in match.group(2)


def test_schema_has_no_case_id_and_is_unique_on_token(connection):
    schema_sql = namespace["SCHEMA_SQL"]

    assert "UNIQUE (token)" in schema_sql
    assert "case_id" not in schema_sql
    assert "CREATE TABLE IF NOT EXISTS vault_schema.issued_tokens" in schema_sql
    assert "CREATE TABLE IF NOT EXISTS metadata_schema.detokenize_log" in schema_sql
    assert "CREATE SCHEMA IF NOT EXISTS vault_schema" in schema_sql
    assert "key_version TEXT NOT NULL" in schema_sql

    ensure_schema(connection)
    assert connection.schema_ready


def test_issued_token_insert_is_idempotent(connection):
    record_issued_token(connection, "[HOST_ABC]", "HOST", 2, "v1")
    record_issued_token(connection, "[HOST_ABC]", "HOST", 2, "v1")

    insert_queries = [query for query, _ in connection.executed]
    assert insert_queries[0].startswith("INSERT INTO VAULT_SCHEMA.ISSUED_TOKENS")
    assert "ON CONFLICT (TOKEN) DO NOTHING" in insert_queries[0]
    assert len(connection.issued) == 1
    assert lookup_issued_token(connection, "[HOST_ABC]") == ("HOST", 2, "v1")

## Stage 1: sub-classification

In [ ]:
# --- stage 1: sub-classification -------------------------------------------
# One record in the shape the EDR corpus actually uses.
EDR_LOG = {
    "class_name": "Detection Finding",
    "severity": "High",
    "action": "Denied",
    "disposition": "Quarantined",
    "message": "Endpoint attempted to connect to a suspicious external IP address",
    "metadata": {
        "uid": "sangfor-df-1788620819072-520",
        "event_code": "1004",
        "log_source": "172.20.28.114 logs[6124]",
        "product": {"vendor_name": "Sangfor", "name": "Endpoint Secure"},
    },
    "device": {
        "hostname": "PC-HR-02",
        "ip": "172.20.28.67",
        "type": "Desktop",
        "agent_list": [{"uid": "sangfor-agent-pc-hr-02",
                        "name": "Sangfor Endpoint Secure Agent"}],
    },
    "finding_info": {
        "title": "Suspicious Network Connection",
        "desc": "chrome.exe initiated an outbound connection to a known malicious IP.",
    },
    "evidences": [{
        "user": {"name": "administrator", "type": "User"},
        "connection": {"dst_endpoint": {"ip": "185.220.101.45", "port": 443},
                       "protocol": "TCP"},
        "process": {"name": "powershell.exe",
                    "cmd_line": "powershell.exe -ExecutionPolicy Bypass"},
        "file": {"name": "update.exe",
                 "path": "C:\\Users\\Public\\Downloads\\update.exe",
                 "hashes": [{"algorithm": "MD5",
                             "value": "8f14e45fceea167a5a36dedd4bea2543"}]},
    }],
}

SERVER_LOG = {
    "class_uid": 2004,
    "device": {"hostname": "SRV-DB-01", "ip": "172.20.28.10", "type": "Server",
               "agent_list": [{"uid": "sangfor-agent-srv-db-01"}]},
    "evidences": [{"user": {"name": "user01"}}],
}

EDR_CII = ["PC-HR-02", "172.20.28.67", "172.20.28.114", "administrator",
           "C:\\Users\\Public\\Downloads", "sangfor-agent-pc-hr-02"]


@pytest.mark.parametrize(
    "hostname, expected",
    [("PC-HR-02", "HOST_DESKTOP"), ("PC-ACCOUNT-03", "HOST_DESKTOP"),
     ("PC-FINANCE-01", "HOST_DESKTOP"), ("PC-SALES-04", "HOST_DESKTOP"),
     ("SRV-DB-01", "HOST_SERVER"), ("fw-edge-1", "HOST")],
)
def test_hostnames_are_sub_classified(hostname, expected):
    assert classify_hostname(hostname) == expected


@pytest.mark.parametrize(
    "username, expected",
    [("administrator", "USER_PRIV"), ("Administrator", "USER_PRIV"),
     ("root", "USER_PRIV"), ("admin", "USER_PRIV"),
     ("user01", "USER"), ("accounting01", "USER"), ("HR-User", "USER"),
     ("dev_ops", "USER"), ("system", "USER")],
)
def test_privileged_users_are_told_apart(username, expected):
    """`system` lands on the ordinary side, which is exactly why this
    hardcoded list belongs in an IAM lookup instead."""
    assert classify_username(username) == expected


def test_the_heuristic_announces_itself():
    reset_heuristic_warning()

    with pytest.warns(UserWarning, match="heuristic"):
        classify_hostname("PC-X")


def test_desktops_and_servers_get_different_prefixes():
    desktop, _ = classify_and_mark(EDR_LOG)
    server, _ = classify_and_mark(SERVER_LOG)

    assert desktop["device"]["hostname"].startswith("[HOST_DESKTOP_")
    assert server["device"]["hostname"].startswith("[HOST_SERVER_")


def test_both_hostname_spellings_are_covered():
    """Some sources write device.name where others write device.hostname."""
    by_name, _ = classify_and_mark({"device": {"name": "PC-HR-02"}})
    by_hostname, _ = classify_and_mark({"device": {"hostname": "PC-HR-02"}})

    assert by_name["device"]["name"] == by_hostname["device"]["hostname"]


def test_privileged_and_ordinary_users_get_different_prefixes():
    privileged, _ = classify_and_mark(EDR_LOG)
    ordinary, _ = classify_and_mark(SERVER_LOG)

    assert privileged["evidences"][0]["user"]["name"].startswith("[USER_PRIV_")
    assert ordinary["evidences"][0]["user"]["name"].startswith("[USER_")
    assert not ordinary["evidences"][0]["user"]["name"].startswith("[USER_PRIV_")


def test_classify_and_mark_does_not_mutate_its_input():
    before = json.dumps(EDR_LOG, sort_keys=True)
    classify_and_mark(EDR_LOG)

    assert json.dumps(EDR_LOG, sort_keys=True) == before

## Stage 1: the file-path codec

In [ ]:



def test_filename_is_never_encrypted():
    """update.exe stays readable for malware analysis."""
    marked = mark_file_path("C:\\Users\\Public\\Downloads\\update.exe")

    assert marked.endswith("\\update.exe")
    assert marked.startswith("[FILE_PATH_USER_PUBLIC_")
    assert "Users" not in marked and "Public" not in marked


@pytest.mark.parametrize(
    "path, separator",
    [("C:\\Users\\Public\\Downloads\\update.exe", "\\"),
     ("C:\\Windows\\System32\\evil.dll", "\\"),
     ("/var/tmp/staging/payload.sh", "/"),
     ("/home/analyst/report.txt", "/")],
)
def test_file_path_round_trips_on_both_separators(path, separator):
    marked = mark_file_path(path)

    assert separator in marked
    assert unmark_cii(marked) == path


def test_a_bare_filename_has_no_directory_to_hide():
    assert mark_file_path("update.exe") == "update.exe"


def test_unrecognised_directories_fall_back_to_the_plain_prefix():
    marked = mark_file_path("D:\\xyz\\thing.bin")

    assert marked.startswith("[FILE_PATH_")
    assert unmark_cii(marked) == "D:\\xyz\\thing.bin"


@pytest.mark.parametrize(
    "value",
    ["C:\\Users\\Public\\Downloads", "172.20.28.114 logs[6124]", "/var/tmp",
     "sangfor-agent-pc-account-03", "a", "  spaced  value  "],
)
def test_the_compact_codec_round_trips_exactly(value):
    assert decode_ascii(encode_ascii(value)) == value
    assert len(encode_ascii(value)) <= MAX_SUFFIX_LEN


def test_a_value_past_the_codec_ceiling_is_refused_loudly():
    with pytest.raises(CiiValueTooLongError):
        mark_file_path("C:\\Users\\Public\\Documents\\Archive\\2026\\x.exe")


def test_non_ascii_is_refused_rather_than_mangled():
    with pytest.raises(NonAsciiValueError):
        encode_ascii("C:\\ผู้ใช้")


def test_every_registered_prefix_resolves_to_a_real_codec():
    for prefix in ["HOST_DESKTOP", "HOST_SERVER", "USER_PRIV", "AGENT",
                   "SRV_LOG_COLLECTOR", *FILE_PATH_PREFIXES]:
        assert PREFIX_CODEC[prefix] in CODECS

## The leak scan

In [ ]:



def test_raw_cii_scan_covers_every_category():
    found = collect_raw_cii(EDR_LOG)

    assert found["hostname"] == {"PC-HR-02"}
    assert found["username"] == {"administrator"}
    assert found["file_directory"] == {"C:\\Users\\Public\\Downloads"}
    assert {"172.20.28.67", "172.20.28.114"} <= found["internal_ip"]
    assert "185.220.101.45" not in found["internal_ip"], "public is not internal"


def test_the_scan_ignores_letter_case():
    """An agent uid spells its host lowercase; a case-sensitive scan would
    call that leak clean."""
    leaks = find_leaks('{"uid": "sangfor-agent-pc-hr-02"}', {"hostname": {"PC-HR-02"}})

    assert leaks == [("hostname", "PC-HR-02")]

## Stages 1-3 end to end

In [ ]:



@pytest.mark.parametrize("document", [EDR_LOG, SERVER_LOG])
def test_no_raw_value_survives_the_pipeline(document, connection, key_ring):
    """The one test this whole pipeline exists to pass."""
    output = json.dumps(sanitize_document(document, key_ring, connection))

    assert not find_leaks(output, collect_raw_cii(document))


def test_the_agent_uid_no_longer_names_its_host(connection, key_ring):
    """sangfor-agent-pc-hr-02 would otherwise hand the hostname straight over."""
    output = sanitize_document(EDR_LOG, key_ring, connection)
    agent_uid = output["device"]["agent_list"][0]["uid"]

    assert agent_uid.startswith("[AGENT_")
    assert "pc-hr-02" not in json.dumps(output).lower()


def test_the_nested_destination_address_is_tokenized(connection, key_ring):
    """evidences[].connection.dst_endpoint.ip is not the top-level path."""
    output = sanitize_document(EDR_LOG, key_ring, connection)
    destination = output["evidences"][0]["connection"]["dst_endpoint"]

    assert destination["ip"].startswith("[EXT_IP_")
    assert destination["port"] == 443


def test_internal_and_external_addresses_keep_different_prefixes(connection, key_ring):
    output = sanitize_document(EDR_LOG, key_ring, connection)

    assert output["device"]["ip"].startswith("[INT_IP_")
    assert output["evidences"][0]["connection"]["dst_endpoint"]["ip"].startswith(
        "[EXT_IP_"
    )


def test_the_filename_is_still_readable_after_all_stages(connection, key_ring):
    output = sanitize_document(EDR_LOG, key_ring, connection)

    assert output["evidences"][0]["file"]["path"].endswith("\\update.exe")
    assert "Public" not in output["evidences"][0]["file"]["path"]
    assert output["evidences"][0]["file"]["name"] == "update.exe"


def test_threat_intelligence_stays_readable(connection, key_ring):
    """These identify malware, not people; the model needs them."""
    output = sanitize_document(EDR_LOG, key_ring, connection)
    evidence = output["evidences"][0]

    assert evidence["process"]["name"] == "powershell.exe"
    assert evidence["file"]["hashes"][0]["value"] == "8f14e45fceea167a5a36dedd4bea2543"
    assert output["finding_info"]["title"] == "Suspicious Network Connection"
    assert output["metadata"]["product"]["vendor_name"] == "Sangfor"


def test_context_an_analyst_needs_is_left_alone(connection, key_ring):
    output = sanitize_document(EDR_LOG, key_ring, connection)

    assert output["severity"] == "High"
    assert output["action"] == "Denied"
    assert output["disposition"] == "Quarantined"
    assert output["device"]["type"] == "Desktop"
    assert output["metadata"]["event_code"] == "1004"


def test_the_command_line_is_swept_as_free_text(connection, key_ring):
    """A cmd_line can name a user or a path, so it is not kept whole."""
    document = json.loads(json.dumps(EDR_LOG))
    document["evidences"][0]["process"]["cmd_line"] = (
        "powershell.exe -Path C:\\Users\\Public\\Downloads"
    )

    marked, _ = classify_and_mark(document)

    assert "C:\\Users\\Public\\Downloads" not in marked["evidences"][0]["process"][
        "cmd_line"
    ]


def test_log_source_is_tokenized_and_deterministic(connection, key_ring):
    """One value repeated across records is one token."""
    first = sanitize_document(EDR_LOG, key_ring, connection)
    second = sanitize_document(EDR_LOG, key_ring, connection)

    assert first["metadata"]["log_source"].startswith("[SRV_LOG_COLLECTOR_")
    assert first["metadata"]["log_source"] == second["metadata"]["log_source"]


def test_the_same_host_is_the_same_token_across_records(connection, key_ring):
    first = sanitize_document(EDR_LOG, key_ring, connection)
    second = sanitize_document(
        {"class_uid": 2004, "device": {"hostname": "PC-HR-02", "type": "Desktop"},
         "severity": "Low"},
        key_ring, connection,
    )

    assert first["device"]["hostname"] == second["device"]["hostname"]


def test_marking_twice_changes_nothing():
    """Stage 1 and mark_cii must both be idempotent."""
    once, _ = classify_and_mark(EDR_LOG)
    once = mark_cii(once, EDR_POLICY)

    twice, _ = classify_and_mark(once)
    assert mark_cii(twice, EDR_POLICY) == once

## The file pipeline

In [ ]:



def test_one_ndjson_line_is_one_ocsf_document():
    assert parse_ocsf_line(json.dumps(EDR_LOG)) == EDR_LOG


def test_a_line_that_is_not_a_document_is_refused():
    """The corpus is already OCSF; anything else is a mistake, not a format."""
    with pytest.raises(NotOcsfError):
        parse_ocsf_line("CEF:0|Sangfor|Endpoint Secure|6.0.4|1003|x|8|src=10.0.0.1")


def test_pipeline_writes_a_sanitized_file_and_checks_it(tmp_path, connection, key_ring):
    source = tmp_path / "in.ndjson"
    source.write_text(
        "\n".join(json.dumps(d) for d in (EDR_LOG, SERVER_LOG)) + "\n", encoding="utf-8"
    )
    destination = tmp_path / "out.log"

    written, raw_values = run_pipeline(
        source, destination, key_ring, connection, verbose=False
    )

    produced = written.read_text(encoding="utf-8").strip().splitlines()
    assert len(produced) == 2
    assert all(json.loads(line) for line in produced)
    assert raw_values["hostname"] == {"PC-HR-02", "SRV-DB-01"}
    assert raw_values["username"] == {"administrator", "user01"}
    assert raw_values["file_directory"] == {"C:\\Users\\Public\\Downloads"}


def test_limit_caps_how_many_records_are_processed(tmp_path, connection, key_ring):
    source = tmp_path / "in.ndjson"
    source.write_text("\n".join([json.dumps(EDR_LOG)] * 5) + "\n", encoding="utf-8")

    written, _ = run_pipeline(
        source, tmp_path / "out.log", key_ring, connection, limit=2, verbose=False
    )

    assert len(written.read_text(encoding="utf-8").strip().splitlines()) == 2


def test_the_pipeline_refuses_to_report_success_on_a_leak(
    tmp_path, connection, key_ring, monkeypatch
):
    """Neutralise Stage 1 and the field rule, then check the scan catches it."""
    source = tmp_path / "in.ndjson"
    source.write_text(json.dumps(EDR_LOG) + "\n", encoding="utf-8")
    # sanitize_document resolves its helpers from the library namespace, so
    # that is what has to be patched.
    monkeypatch.setitem(
        namespace,
        "classify_and_mark",
        lambda document, emitted=None, policy=None: (document, {}),
    )
    monkeypatch.delitem(FIELD_RULES, "device.hostname", raising=False)

    with pytest.raises(RawCiiLeakError) as error:
        run_pipeline(source, tmp_path / "out.log", key_ring, connection, verbose=False)

    assert ("hostname", "PC-HR-02") in error.value.findings


def test_the_real_corpus_sanitizes_cleanly(connection, key_ring):
    """A live sample of Data/ocsf_edr_mock.ndjson, when it is present."""
    if not OCSF_INPUT.exists():
        pytest.skip(f"{OCSF_INPUT} is not present")

    with OCSF_INPUT.open(encoding="utf-8") as handle:
        documents = [json.loads(line) for _, line in zip(range(50), handle)]

    for document in documents:
        output = json.dumps(sanitize_document(document, key_ring, connection))
        assert not find_leaks(output, collect_raw_cii(document))

## The detection finding schema

In [ ]:
# --- the detection finding schema ------------------------------------------
# The record the connector actually delivers: OCSF class_uid 2004, already
# parsed. Its `raw_data` repeats every CII value the structured fields carry,
# which is the thing this round has to get right.
FINDING = {
    "activity_id": 1, "activity_name": "Create",
    "category_name": "Findings", "category_uid": 2,
    "class_name": "Detection Finding", "class_uid": 2004,
    "device": {
        "hostname": "PC-HR-02", "ip": "172.20.28.67",
        "is_isolated": True, "type": "Desktop", "type_id": 2,
    },
    "evidences": [{
        "data": {"affected_files_count": 24, "behavior": "Mass file encryption"},
        "process": {
            "name": "powershell.exe", "uid": "powershell.exe",
            "user": {"name": "HR-User"},
        },
    }],
    "finding_info": {
        "desc": "Ransomware-like behavior detected, malicious process "
                "terminated and endpoint isolated",
        "title": "Ransomware Behavior Detection", "uid": "1002",
    },
    "is_alert": True,
    "metadata": {"product": {"name": "Endpoint Secure", "vendor_name": "Sangfor",
                             "version": "6.0.4"}, "version": "1.8.0"},
    "raw_data": (
        "2026-09-03T00:23:12+07:00 172.20.28.114 logs[6124]: CEF:0|Sangfor|"
        "Endpoint Secure|6.0.4|1002|Ransomware Behavior Detection|10|"
        "src=172.20.28.67 suser=HR-User shost=PC-HR-02 act=Block "
        "cs1Label=Process cs1=powershell.exe cs2Label=Behavior "
        "cs2=Mass file encryption cs3Label=AffectedFiles cs3=24 "
        "cs4Label=Isolation cs4=Enabled msg=Ransomware-like behavior "
        "detected, malicious process terminated and endpoint isolated"
    ),
    "severity": "Medium", "severity_id": 3,
    "status": "New", "status_detail": "Block", "status_id": 1,
    "time": 1788369792000, "timezone_offset": 420,
    "type_name": "Detection Finding: Create", "type_uid": 200401,
}

FINDING_CII = ["PC-HR-02", "172.20.28.67", "HR-User", "172.20.28.114"]

# Everything acceptance criterion 4 says must come through untouched.
UNTOUCHED_PATHS = [
    ("evidences", 0, "process", "name"),
    ("evidences", 0, "process", "uid"),
    ("evidences", 0, "data", "behavior"),
    ("evidences", 0, "data", "affected_files_count"),
    ("finding_info", "title"),
    ("finding_info", "uid"),
    ("device", "is_isolated"),
    ("device", "type"),
    ("device", "type_id"),
]


def at(document, path):
    node = document
    for key in path:
        node = node[key]
    return node

## Stage 0: validation

In [ ]:



def test_a_well_formed_finding_passes_validation():
    assert validate_ocsf(FINDING) is FINDING


def test_a_record_with_no_class_is_refused():
    with pytest.raises(OcsfValidationError, match="class_uid"):
        validate_ocsf({"device": {"hostname": "PC-HR-02"}})


@pytest.mark.parametrize(
    "broken",
    [{"class_uid": 2004, "evidences": {"process": {}}},
     {"class_uid": 2004, "device": "PC-HR-02"},
     {"class_uid": 2004, "metadata": []}],
)
def test_a_container_of_the_wrong_type_is_refused(broken):
    """The marker walks containers; a scalar where a list belongs would be
    stepped over and the output would look clean while carrying raw values."""
    with pytest.raises(OcsfValidationError):
        validate_ocsf(broken)


def test_validation_runs_before_anything_is_marked(connection, key_ring):
    with pytest.raises(OcsfValidationError):
        sanitize_document({"class_uid": 2004, "device": "PC-HR-02"},
                          key_ring, connection)

## Acceptance 1: device.type decides, not the hostname

In [ ]:



def test_the_host_prefix_comes_from_device_type(connection, key_ring):
    """Acceptance 1."""
    output = sanitize_document(FINDING, key_ring, connection)

    assert output["device"]["hostname"].startswith("[HOST_DESKTOP_")


def test_device_type_beats_a_hostname_that_disagrees():
    """Acceptance 1, proved: the name says nothing, the record says Desktop."""
    assert classify_hostname("srv-legacy-box", {"type": "Desktop"}) == "HOST_DESKTOP"
    assert classify_hostname("PC-HR-02", {"type": "Server"}) == "HOST_SERVER"
    assert classify_hostname("anything", {"type": "Desktop"}) == "HOST_DESKTOP"


def test_type_id_is_used_when_the_label_is_missing():
    assert classify_hostname("anything", {"type_id": 2}) == "HOST_DESKTOP"
    assert classify_hostname("anything", {"type_id": 1}) == "HOST_SERVER"


def test_the_hostname_guess_is_only_a_fallback():
    """Network Activity records carry no device.type, so the guess stays."""
    assert classify_hostname("PC-HR-02", {}) == "HOST_DESKTOP"
    assert classify_hostname("SRV-DB-01", None) == "HOST_SERVER"
    assert classify_hostname("fw-edge-1", {}) == "HOST"


def test_only_the_guess_warns():
    """A record that states its type is not a heuristic and must not say so."""
    reset_heuristic_warning()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        classify_hostname("anything", {"type": "Desktop"})
    assert not caught

    reset_heuristic_warning()
    with pytest.warns(UserWarning, match="heuristic"):
        classify_hostname("PC-HR-02", {})


def test_an_unknown_device_type_falls_through_to_the_guess():
    assert classify_hostname("PC-HR-02", {"type": "Toaster"}) == "HOST_DESKTOP"
    assert classify_hostname("fw-1", {"type": "Firewall", "type_id": 9}) == "HOST"

## Acceptance 2 and 3: addresses and users

In [ ]:



def test_the_private_device_address_is_internal(connection, key_ring):
    """Acceptance 2."""
    output = sanitize_document(FINDING, key_ring, connection)

    assert output["device"]["ip"].startswith("[INT_IP_")


def test_the_nested_process_user_is_tokenized(connection, key_ring):
    """Acceptance 3: evidences[].process.user.name is its own path."""
    output = sanitize_document(FINDING, key_ring, connection)
    name = at(output, ("evidences", 0, "process", "user", "name"))

    assert name.startswith("[USER_")
    assert not name.startswith("[USER_PRIV_"), "HR-User is not privileged"


def test_a_privileged_nested_user_still_gets_the_privileged_prefix():
    document = json.loads(json.dumps(FINDING))
    document["evidences"][0]["process"]["user"]["name"] = "administrator"

    marked, _ = classify_and_mark(document)

    assert at(marked, ("evidences", 0, "process", "user", "name")).startswith(
        "[USER_PRIV_"
    )

## Acceptance 4: what must not be touched

In [ ]:



@pytest.mark.parametrize("path", UNTOUCHED_PATHS)
def test_non_cii_fields_come_through_byte_for_byte(path, connection, key_ring):
    """Acceptance 4: these say what happened, not who or where."""
    output = sanitize_document(FINDING, key_ring, connection)

    assert at(output, path) == at(FINDING, path)


def test_the_whole_record_changes_only_where_it_must(connection, key_ring):
    output = sanitize_document(FINDING, key_ring, connection)

    changed = sorted(key for key in FINDING if FINDING[key] != output[key])
    assert changed == ["device", "evidences", "raw_data"]

## Acceptance 5: raw_data, the one that matters most

In [ ]:



@pytest.mark.parametrize("secret", FINDING_CII)
def test_raw_data_keeps_no_raw_value(secret, connection, key_ring):
    """Acceptance 5, one substring at a time so a failure names the value."""
    raw_data = sanitize_document(FINDING, key_ring, connection)["raw_data"]

    assert secret not in raw_data


def test_raw_data_uses_the_same_tokens_as_the_structured_fields(connection, key_ring):
    """Substituting different tokens would break correlation for the model."""
    output = sanitize_document(FINDING, key_ring, connection)

    assert output["device"]["hostname"] in output["raw_data"]
    assert output["device"]["ip"] in output["raw_data"]
    assert at(output, ("evidences", 0, "process", "user", "name")) in output["raw_data"]


def test_raw_data_keeps_everything_that_is_not_cii(connection, key_ring):
    raw_data = sanitize_document(FINDING, key_ring, connection)["raw_data"]

    for kept in ["CEF:0|Sangfor|Endpoint Secure|6.0.4|1002", "act=Block",
                 "cs1=powershell.exe", "cs2=Mass file encryption",
                 "cs3=24", "cs4=Enabled"]:
        assert kept in raw_data


def test_the_collector_address_in_raw_data_is_tokenized(connection, key_ring):
    """172.20.28.114 appears only in raw_data, never in a structured field."""
    raw_data = sanitize_document(FINDING, key_ring, connection)["raw_data"]

    assert "172.20.28.114" not in raw_data
    assert "[INT_IP_" in raw_data


def test_the_leak_scan_agrees_with_the_substring_check(connection, key_ring):
    output = json.dumps(sanitize_document(FINDING, key_ring, connection))

    assert not find_leaks(output, collect_raw_cii(FINDING))
    assert not [secret for secret in FINDING_CII if secret in output]

## Acceptance 6: finding_info.desc

In [ ]:



def test_the_description_still_goes_through_the_free_text_sweep():
    """Acceptance 6. This record's desc names nothing, so use one that does."""
    document = json.loads(json.dumps(FINDING))
    document["finding_info"]["desc"] = (
        "HR-User on PC-HR-02 (172.20.28.67) triggered ransomware behavior"
    )

    # Stage 1 sweeps the values it marked; device.ip is mark_cii's, so the
    # description only comes out clean once both passes have run.
    marked, _ = classify_and_mark(document)
    desc = mark_cii(marked, EDR_POLICY)["finding_info"]["desc"]

    for secret in ["HR-User", "PC-HR-02", "172.20.28.67"]:
        assert secret not in desc
    assert "triggered ransomware behavior" in desc


def test_the_untouched_description_survives_intact(connection, key_ring):
    output = sanitize_document(FINDING, key_ring, connection)

    assert output["finding_info"]["desc"] == FINDING["finding_info"]["desc"]

## Run the tests

In [ ]:
import subprocess
import sys
import tempfile


def build_test_module():
    """Concatenate the cells of this notebook that are tagged 'test'."""
    this_notebook = locate_notebook("test_encryp.ipynb")
    cells = json.loads(this_notebook.read_text(encoding="utf-8"))["cells"]
    tagged = [
        cell
        for cell in cells
        if cell["cell_type"] == "code" and "test" in cell["metadata"].get("tags", [])
    ]
    if not tagged:
        raise AssertionError("no code cell in test_encryp.ipynb is tagged 'test'")
    return "\n\n".join("".join(cell["source"]) for cell in tagged)


def run_tests(*pytest_args):
    """Run the tagged cells under real pytest and print its report.

    Reads this notebook from disk, so save it before running. Pass extra
    pytest arguments through, e.g. run_tests("-k", "raw_data", "-v").

    Both ends of the pipe are pinned to UTF-8. `text=True` would decode with
    the locale encoding instead - cp874 on a Thai Windows install - and the
    Thai heuristic warning in pytest's output would crash the reader thread.
    """
    with tempfile.TemporaryDirectory() as directory:
        module_path = Path(directory) / "test_encryp_cells.py"
        module_path.write_text(build_test_module(), encoding="utf-8")
        completed = subprocess.run(
            [sys.executable, "-m", "pytest", str(module_path), "-q",
             "-p", "no:cacheprovider", *pytest_args],
            capture_output=True,
            encoding="utf-8",
            errors="replace",
            env={
                **os.environ,
                "ENCRYP_NOTEBOOK": str(NOTEBOOK.resolve()),
                "PYTHONIOENCODING": "utf-8",
            },
        )
    print(completed.stdout or completed.stderr)
    return completed.returncode


run_tests()